# Phase 1: Evaluation — RLVR Model vs Baseline

Comparing our GRPO-trained model against the unmodified base model 
on 100 unseen GSM8K test problems.

## What We're Testing
- **Trained model**: Qwen2.5-1.5B after 500 steps of GRPO on 500 GSM8K examples
- **Base model**: Qwen2.5-1.5B-Instruct with no RL training
- **Benchmark**: GSM8K test split (problems never seen during training)
- **Metric**: Pass@1 accuracy — is the final answer exactly correct?

In [8]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps xformers

## Evaluation Setup

Both models use identical:
- System prompt (think in `<think>` tags, answer in `<answer>` tags)
- Temperature: 0.1 (near-deterministic for fair comparison)
- Max new tokens: 512
- Answer extraction: last number found in `<answer>` tags

The only difference is the model weights.

In [21]:
import re
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

# ==========================================
# 1. LOAD YOUR TRAINED MODEL
# ==========================================
model_path = "/kaggle/input/notebooks/pasha1701/rlproject/grpo_gsm8k/checkpoint-500"

print(f"Loading trained model from: {model_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# ==========================================
# 2. EVALUATION LOGIC
# ==========================================
SYSTEM_PROMPT = "You are a math reasoning assistant. Always think through problems step by step inside <think> tags. Then give your final answer inside <answer> tags."

def evaluate_single_model(model, tokenizer, test_dataset, num_problems=100):
    correct = 0
    format_failures = 0
    
    for i in range(num_problems):
        problem = test_dataset[i]['question']
        gt = test_dataset[i]['answer'].split('####')[1].strip().replace(',','')
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem}
        ]
        
        text_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        text_prompt += "<think>\n"
        
        inputs = tokenizer([text_prompt], return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=512, 
                use_cache=True, temperature=0.1,  # low temp = deterministic
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
        matches = re.findall(r'<answer>(.*?)</answer>', response, re.DOTALL)
        
        if matches:
            numbers = re.findall(r'-?\d+\.?\d*', matches[-1].strip())
            predicted = numbers[-1].replace(',','') if numbers else None
        else:
            predicted = None
            format_failures += 1
        
        if predicted == gt:
            correct += 1
        
        if (i + 1) % 10 == 0:
            print(f"Step {i+1}/100 | Running accuracy: {correct}/{i+1} = {correct/(i+1)*100:.1f}%")
    
    print(f"\n{'='*40}")
    print(f"Final Score: {correct}/100 = {correct}%")
    print(f"Format failures: {format_failures}/100")
    return correct

# ==========================================
# 3. RUN THE TEST
# ==========================================
test_data = load_dataset("openai/gsm8k", "main", split="test")
print("Starting 100-problem evaluation...")
your_score = evaluate_single_model(model, tokenizer, test_data, 100)

Loading trained model from: /kaggle/input/notebooks/pasha1701/rlproject/grpo_gsm8k/checkpoint-500...
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Starting 100-problem evaluation...
Step 10/100 | Running accuracy: 6/10 = 60.0%
Step 20/100 | Running accuracy: 12/20 = 60.0%
Step 30/100 | Running accuracy: 19/30 = 63.3%
Step 40/100 | Running accuracy: 21/40 = 52.5%
Step 50/100 | Running accuracy: 27/50 = 54.0%
Step 60/100 | Running accuracy: 34/60 = 56.7%
Step 70/100 | Running accuracy: 37/70 = 52.9%
Step 80/100 | Running accuracy: 44/80 = 55.0%
Step 90/100 | Running accuracy: 51/90 = 56.7%
Step 100/100 | Running accuracy: 58/100 = 58.0%

Final Score: 58/100 = 58%
Format failures: 8/100


## Trained Model Results (above)

The RLVR model achieves **58% accuracy** on 100 unseen problems.

Key observations from the outputs:
- Model consistently uses `<think>` tags for structured reasoning
- Only 8/100 format failures (model knows how to answer)
- Reasoning chains show step-by-step decomposition

Now evaluating the base model on the same 100 problems for comparison.

## Base Model Results (above)

The unmodified base model achieves **42% accuracy** on the same 100 problems.
17/100 format failures — model often doesn't use `<answer>` tags consistently.

In [22]:
import re
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

# ==========================================
# 1. LOAD THE UNTRAINED BASE MODEL
# ==========================================
model_name = "unsloth/Qwen2.5-1.5B-Instruct"

print(f"Loading base model: {model_name}...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

# ==========================================
# 2. EVALUATION LOGIC
# ==========================================
SYSTEM_PROMPT = "You are a math reasoning assistant. Always think through problems step by step inside <think> tags. Then give your final answer inside <answer> tags."

def evaluate_base_model(model, tokenizer, test_dataset, num_problems=100):
    correct = 0
    format_failures = 0
    
    for i in range(num_problems):
        problem = test_dataset[i]['question']
        gt = test_dataset[i]['answer'].split('####')[1].strip().replace(',','')
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem}
        ]
        
        text_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        text_prompt += "<think>\n"
        
        inputs = tokenizer([text_prompt], return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=512, 
                use_cache=True, temperature=0.1,  
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
        matches = re.findall(r'<answer>(.*?)</answer>', response, re.DOTALL)
        
        if matches:
            numbers = re.findall(r'-?\d+\.?\d*', matches[-1].strip())
            predicted = numbers[-1].replace(',','') if numbers else None
        else:
            predicted = None
            format_failures += 1
        
        if predicted == gt:
            correct += 1
        
        if (i + 1) % 10 == 0:
            print(f"Step {i+1}/100 | Base model accuracy: {correct}/{i+1} = {correct/(i+1)*100:.1f}%")
    
    print(f"\n{'='*40}")
    print(f"Base Model Final Score: {correct}/100 = {correct}%")
    print(f"Base Model Format failures: {format_failures}/100")
    return correct

# ==========================================
# 3. RUN THE TEST
# ==========================================
test_data = load_dataset("openai/gsm8k", "main", split="test")
print("Starting 100-problem evaluation on the BASE model...")
base_score = evaluate_base_model(base_model, base_tokenizer, test_data, 100)

Loading base model: unsloth/Qwen2.5-1.5B-Instruct...
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Starting 100-problem evaluation on the BASE model...
Step 20/100 | Base model accuracy: 5/20 = 25.0%
Step 30/100 | Base model accuracy: 10/30 = 33.3%
Step 40/100 | Base model accuracy: 13/40 = 32.5%
Step 50/100 | Base model accuracy: 18/50 = 36.0%
Step 60/100 | Base model accuracy: 23/60 = 38.3%
Step 70/100 | Base model accuracy: 26/70 = 37.1%
Step 80/100 | Base model accuracy: 32/80 = 40.0%
Step 90/100 | Base model accuracy: 36/90 = 40.0%
Step 100/100 | Base model accuracy: 42/100 = 42.0%

Base Model Final Score: 42/100 = 42%
Base Model Format failures: 17/100


## Final Comparison

| Model | GSM8K Accuracy | Format Failures |
|-------|---------------|-----------------|
| Qwen2.5-1.5B-Instruct (base) | 42% | 17/100 |
| + GRPO/RLVR (ours, 500 steps) | **58%** | 8/100 |
| **Improvement** | **+16pts (+38% relative)** | **-53%** |

**Trained on:** 500 GSM8K examples  
**Training time:** ~3.5 hours  
**Hardware:** 2x Kaggle T4 GPUs (free)  
**Cost:** $0

### Why This Works
The "aha moment" at step 70 (reward jumped 0.375 → 0.719) shows the model 
spontaneously learned structured reasoning through pure RL signal — 
no human-labeled reasoning traces needed.

This reproduces the core finding of DeepSeek-R1 (arXiv:2501.12948) 
at small scale on free hardware.